# Reducto Extract to Oracle JSON Demo

This notebook demonstrates Reducto `/extract` for schema-typed JSON, then stores the whole extraction result in Oracle Database. Use `/parse` ingestion when you want chunks, vectors, tables, and promoted facts. Use `/extract` when you already know the fields your application needs.

In [2]:
import os
import json

from reducto.lib.oracledb.utils import read_lob
from reducto.lib.oracledb.config import connect_oracle
from reducto.lib.oracledb.models import DocumentMetadata
from reducto.lib.oracledb.oracle import OracleSchemaManager, OracleDocumentRepository
from reducto.lib.oracledb.reducto_client import ReductoDocumentParser

DOCUMENT_URL = "https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm"

required_env = ["REDUCTO_API_KEY", "ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN"]
missing = [name for name in required_env if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Set these environment variables before running: {', '.join(missing)}")

## Define the typed schema

The schema tells Reducto exactly which fields to pull from the document. With citations enabled, each extracted value can include source evidence.

In [3]:
financial_extract_schema = {
    "type": "object",
    "properties": {
        "company_name": {"type": "string"},
        "fiscal_year": {"type": "integer"},
        "filing_type": {"type": "string"},
        "total_net_sales_millions": {"type": "number"},
        "net_income_millions": {"type": "number"},
        "cash_and_cash_equivalents_millions": {"type": "number"},
        "auditor_name": {"type": "string"},
        "business_summary": {"type": "string"},
    },
    "required": ["company_name", "fiscal_year", "filing_type"],
}

print(json.dumps(financial_extract_schema, indent=2))

{
  "type": "object",
  "properties": {
    "company_name": {
      "type": "string"
    },
    "fiscal_year": {
      "type": "integer"
    },
    "filing_type": {
      "type": "string"
    },
    "total_net_sales_millions": {
      "type": "number"
    },
    "net_income_millions": {
      "type": "number"
    },
    "cash_and_cash_equivalents_millions": {
      "type": "number"
    },
    "auditor_name": {
      "type": "string"
    },
    "business_summary": {
      "type": "string"
    }
  },
  "required": [
    "company_name",
    "fiscal_year",
    "filing_type"
  ]
}


## Run Reducto Extract

This calls `client.extract.run(input=..., instructions={"schema": ...}, settings={"citations": {"enabled": True}})` through the project wrapper.

In [4]:
parser = ReductoDocumentParser()
extract_result = parser.extract_url(
    DOCUMENT_URL,
    schema=financial_extract_schema,
    citations=True,
    deep_extract=False,
    system_prompt="Extract audited annual filing fields. Preserve units in field names where possible.",
)

print("Job:", extract_result.job_id)
print("Studio:", extract_result.studio_link)
print(json.dumps(extract_result.extracted_json, indent=2)[:3000])

Job: 5086c296-e13f-48a1-a90a-d80607c6ca7f
Studio: https://studio.reducto.ai/job/5086c296-e13f-48a1-a90a-d80607c6ca7f
{
  "filing_type": {
    "value": "FORM 10-K",
    "citations": [
      {
        "type": "Text",
        "bbox": {
          "left": 0.1045734126984127,
          "top": 0.3751630686750315,
          "width": 0.7908531746031746,
          "height": 0.01941346105367915,
          "page": 5,
          "original_page": 5
        },
        "content": "FORM 10-K",
        "image_url": null,
        "chart_data": null,
        "confidence": "high",
        "granular_confidence": {
          "extract_confidence": 0.0,
          "parse_confidence": null
        },
        "extra": null
      }
    ]
  },
  "fiscal_year": {
    "value": 2023,
    "citations": [
      {
        "type": "Text",
        "bbox": {
          "left": 0.1045734126984127,
          "top": 0.5338845335407021,
          "width": 0.7908531746031746,
          "height": 0.01941346105367915,
          "page

## Store the whole extraction JSON in Oracle

`DOCUMENTS.raw_reducto_output` keeps the full raw Reducto response. `DOCUMENT_EXTRACTIONS` stores the schema, extracted typed JSON, raw response, and citation setting.

In [ ]:
connection = connect_oracle()
OracleSchemaManager(connection).create_schema(vector_dimensions=int(os.getenv("ORACLE_VECTOR_DIMENSIONS", "384")))

metadata = DocumentMetadata(
    source_uri=DOCUMENT_URL,
    source_kind="url",
    company="AAPL",
    fiscal_year=2023,
    filing_type="10-K",
    title="Apple 2023 Form 10-K Extract",
)

document_id, extraction_id = OracleDocumentRepository(connection).store_extract_result(
    metadata,
    extract_result,
)

print({"document_id": document_id, "extraction_id": extraction_id})

## Read it back

Use `JSON_SERIALIZE` so the Python driver returns canonical JSON text, preserving numeric values when we load it back with `json.loads`.

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT JSON_SERIALIZE(extracted_json RETURNING CLOB)
        FROM document_extractions
        WHERE extraction_id = :extraction_id
        """,
        extraction_id=extraction_id,
    )
    row = cursor.fetchone()

stored_payload = read_lob(row[0])
if isinstance(stored_payload, str):
    stored_payload = json.loads(stored_payload)

print(json.dumps(stored_payload, indent=2)[:3000])

## CLI equivalent

Save the schema JSON to a file and run:

```bash
reducto-oracledb ingest \
  --mode extract \
  --url "https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm" \
  --schema-file ./schemas/financial_extract_schema.json \
  --company AAPL \
  --year 2023 \
  --filing-type 10-K \
  --title "Apple 2023 Form 10-K Extract"
```